In [ ]:
!pip install pandas matplotlib gdown

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import gdown
import os

In [ ]:
# Download data from Google Sheets using gdown
sheet_id = '1e_lKct9ovnYByYkGpFCKrMpKQs5TQdKhTld9tDU1JcQ'
gid_baseline = '1387240715'
gid_fractal = '1558614809'
url_baseline = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid_baseline}'
url_fractal = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid_fractal}'

baseline_csv = 'baseline_data.csv'
fractal_csv = 'fractal_data.csv'

gdown.download(url_baseline, baseline_csv, quiet=False)
gdown.download(url_fractal, fractal_csv, quiet=False)

In [ ]:
# Load dataset
df_baseline = pd.read_csv(baseline_csv)
df_baseline

In [ ]:
# Load dataset
df_fractal = pd.read_csv(fractal_csv)
df_fractal

In [ ]:
# Clean up df_baseline
df_baseline.columns = df_baseline.columns.astype(str).str.strip().str.lower().str.replace(' ', '_')
df_baseline = df_baseline.loc[:, ~df_baseline.columns.duplicated()]

df_baseline

In [ ]:
# Clean up df_fractal
df_fractal.columns = df_fractal.columns.astype(str).str.strip().str.lower().str.replace(' ', '_')

df_fractal['metaheuristic'] = df_fractal['metaheuristic'].replace({'ts1': 'ts'})

# Divide the fractal score and its CI by 3 ONLY for 'triangledensestsubgraph'
tds_mask = df_fractal['objective_function'] == 'triangledensestsubgraph'

df_fractal.loc[tds_mask, 'average_of_cost_best_solution'] /= 3

if 'ci_cost' in df_fractal.columns:
    df_fractal.loc[tds_mask, 'ci_cost'] /= 3

# Drop duplicates and unnamed garbage columns from pivot table exports

df_fractal = df_fractal.loc[:, ~df_fractal.columns.duplicated()]
df_fractal = df_fractal.loc[:, ~df_fractal.columns.str.contains('unnamed')]

# Sort by time so the lines draw correctly from left to right on the plot
df_fractal.sort_values(by=['graph_name', 'objective_function', 'metaheuristic', 'average_of_total_time_ms'], inplace=True)

df_fractal

In [ ]:
# Define mapping and styling
baseline_mapping = {
    'densesubgraph': ('Greedy Peeling (Greedy++)', 'Greedy++'),
    'triangledensestsubgraph': ('Triangle Densest Batch Peeling', 'BatchGreedyPeeling')
}

graphs = ['citeseer', 'amazon', 'dblp', 'patents', 'livejournal', 'youtube']
metaheuristics = df_fractal['metaheuristic'].unique()
colors = {
    'ts': '#9900FF',
    'ils': '#FE7033',
    'vns': '#32CD32'
}
markers = {'vns': 'o', 'ils': 's', 'ts': '^'}

In [ ]:
# Generate plots
for obj_func, (search_alg, display_alg) in baseline_mapping.items():
    for graph in graphs:
        # Create a single figure for each graph
        fig, ax = plt.subplots(figsize=(8, 6))

        # Set Titles
        ax.set_xlabel('Total Time (ms) - Log Scale', fontsize=18)
        ax.set_ylabel('Score Ratio (MH / Baseline)', fontsize=18)

        ax.set_xlim(10**0, 10**9)
        ax.set_ylim(0, 2)

        # Make the axis numbers larger
        ax.tick_params(axis='both', which='major', labelsize=14)

        # Use log scale for X due to massive differences in runtime
        ax.set_xscale('log')

        # Plot Baseline
        baseline_data = df_baseline[(df_baseline['objective_function'] == obj_func) &
                                    (df_baseline['algorithm'] == search_alg) &
                                    (df_baseline['graph'] == graph)]

        if not baseline_data.empty:
            baseline_score = baseline_data['cost_best_solution'].values[0]
            baseline_time = baseline_data['total_time_(ms)'].values[0]

            # Plot Baseline at exactly Y = 1.0
            ax.scatter(baseline_time, 1.0, color='red', marker='*', s=250, label=display_alg, zorder=5)

            # Plot Metaheuristics
            for meta in metaheuristics:
                meta_data = df_fractal[(df_fractal['objective_function'] == obj_func) &
                                       (df_fractal['metaheuristic'] == meta) &
                                       (df_fractal['graph_name'] == graph)]

                if not meta_data.empty:
                    normalized_score = meta_data['average_of_cost_best_solution'] / baseline_score

                    # Account for CI if it exists in this dataset
                    if 'ci_cost' in meta_data.columns:
                        normalized_ci = meta_data['ci_cost'] / baseline_score
                    else:
                        normalized_ci = 0

                    # Plot with minimalist error bars
                    ax.errorbar(
                        meta_data['average_of_total_time_ms'],
                        normalized_score,
                        yerr=normalized_ci,
                        color=colors.get(meta, 'black'),
                        marker=markers.get(meta, 'o'),
                        markersize=8,
                        linestyle='-',
                        linewidth=2,
                        elinewidth=1,
                        capsize=0,
                        alpha=0.9,
                        label=f"{meta.upper()}"
                    )

        ax.grid(True, which="major", ls="--", alpha=0.3)

        # Legend with a slightly larger font to match the new style
        ax.legend(loc='best', fontsize=15)

        plt.tight_layout()

        # Define dir and file name
        output_folder = "pareto_plots/quality_runtime"
        os.makedirs(output_folder, exist_ok=True)
        filename = f'quality_runtime_{graph}_{obj_func}_pareto.pdf'
        filepath = os.path.join(output_folder, filename)

        # Save figure
        plt.savefig(filepath, bbox_inches='tight')

        plt.show()